# RQ1: Arabic-focused LLM classifier (Jais 2 8B Chat) on Colab

Zero-shot problem-type classification with **Jais 2 8B Chat** (`inception42/Jais-2-8B-Chat`, Apache-2.0, MBZUAI / Inception / Cerebras), using the same labelling guide and test questions as `src/models/llm_classifier.py` (Qwen3-8B via Ollama).

**Before running**
1. *Runtime → Change runtime type → T4 GPU* (free tier is enough: the model is loaded in 4-bit, ~6 GB).
2. On Hugging Face, open https://huggingface.co/inception42/Jais-2-8B-Chat, log in and accept the terms (the model is gated).
3. Create a Hugging Face access token (read) and add it in Colab: *🔑 Secrets → name `HF_TOKEN`*. Do not paste the token into a cell.
4. Upload `colab/rq1_test.csv` and `colab/labelling_guide.md` (made by `python src/models/export_for_colab.py`) with the 📁 Files panel.

**After running**: download `rq1_llm_jais-2-8b_cache.csv`, put it in the repo's `results/` folder and run
`python src/models/llm_classifier.py --model jais-2-8b --cache-only` to score it with the shared RQ1 scoring.

Differences from the Ollama run (mention in the paper): 4-bit quantisation via bitsandbytes; no JSON-schema constraint, so the label is parsed from the reply (up to 3 tries; tries 2-3 sample at temperature 0.3).

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))  # reads the Colab secret; the token is never printed

In [ ]:
import os
import pandas as pd

test = pd.read_csv("rq1_test.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
guide = open("labelling_guide.md", encoding="utf-8").read()
print(len(test), "questions;", test["lang"].value_counts().to_dict())

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "inception42/Jais-2-8B-Chat"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_quant_type="nf4"),
    device_map="auto",
)
model.eval()
print("loaded on", model.device)

In [ ]:
import re

LABELS = ["unpaid_wages", "passport_withheld", "gratuity", "unfair_termination",
          "recruitment_fees", "court_enforcement", "other"]

def parse_label(reply: str):
    """First allowed label in the model's reply (accepts 'unpaid wages' too), else None."""
    text = reply.lower().replace("-", "_")
    hits = []
    for label in LABELS:
        for form in (label, label.replace("_", " ")):
            m = re.search(r"\b" + re.escape(form) + r"\b", text)
            if m:
                hits.append((m.start(), label))
    return min(hits)[1] if hits else None

INSTRUCTIONS = """You classify questions that people in the UAE posted about employment problems.
Read the question and choose its problem_type, following the labelling guide below exactly.
Answer with ONLY one of these values and nothing else: """ + ", ".join(LABELS) + """

<labelling_guide>
""" + guide + """
</labelling_guide>"""

def ask(text: str, attempt: int) -> str:
    # instructions go in the user turn: not every chat template supports a system role
    messages = [{"role": "user", "content": INSTRUCTIONS + "\n\n<question>\n" + text + "\n</question>"}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                           return_dict=True).to(model.device)
    sampling = dict(do_sample=True, temperature=0.3) if attempt > 1 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, pad_token_id=tokenizer.eos_token_id, **sampling)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def classify(text: str, tries: int = 3):
    for attempt in range(1, tries + 1):
        label = parse_label(ask(text, attempt))
        if label:
            return label
    return None  # left out of the cache, so it is retried on the next run

# quick look at one Arabic and one English question
for _, q in test.groupby("lang").head(1).iterrows():
    print(q["lang"], q["question_id"], "->", repr(ask(q["text"], 1)))

In [ ]:
import time

CACHE = "rq1_llm_jais-2-8b_cache.csv"  # same format as the local cache: question_id, pred
cache = {}
if os.path.exists(CACHE):  # resume after a disconnect
    c = pd.read_csv(CACHE, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    cache = dict(zip(c["question_id"], c["pred"]))

todo = test[~test["question_id"].isin(cache)]
print(len(cache), "cached,", len(todo), "to classify")
start = time.time()
for i, q in enumerate(todo.itertuples(), 1):
    label = classify(q.text)
    if label is None:
        print(f"[{i}/{len(todo)}] {q.question_id}: no valid label after 3 tries (retried next run)")
        continue
    cache[q.question_id] = label
    pd.DataFrame({"question_id": list(cache), "pred": list(cache.values())}).to_csv(
        CACHE, index=False, encoding="utf-8-sig")  # save after every question
    print(f"[{i}/{len(todo)}] {q.question_id} {q.lang}: {label}  ({(time.time() - start) / i:.1f}s each)")
print(len(cache), "of", len(test), "done")

In [ ]:
from google.colab import files
files.download(CACHE)  # put it in the repo's results/ folder